# Reproject Seagrass Data and Add Standardized Fields

This notebook processes all shapefiles and layers in File Geodatabases (`.gdb`) across all data folders under the notebook root:

1. Reads the list of standard fields from `Standardized Seagrass Attribute Table.xlsx`.
2. Adds any missing fields based on `Column Name (snake_case)` and `Data Type`.
3. Reprojects geometries to `EPSG:4326` / WGS 84.
4. Writes results to a new output folder without modifying the original data.

Important note: the default output is `GeoPackage (.gpkg)` because shapefiles only support field names up to 10 characters. If output is forced to shapefile, many standard field names will be truncated.

In [1]:
# If the libraries are not available yet, run this cell once and restart the kernel if prompted.
# On some Windows installations, geopandas is most stable when installed through conda/mamba.

#%pip install pandas geopandas pyogrio openpyxl shapely fiona

In [2]:
from pathlib import Path
import re
import warnings

import pandas as pd
import geopandas as gpd

try:
    import pyogrio
except Exception:
    pyogrio = None

try:
    import fiona
except Exception:
    fiona = None

warnings.filterwarnings("default")
print("pandas", pd.__version__)
print("geopandas", gpd.__version__)
print("pyogrio", getattr(pyogrio, "__version__", "not installed"))
print("fiona", getattr(fiona, "__version__", "not installed"))

pandas 2.3.3
geopandas 1.1.1
pyogrio 0.11.0
fiona not installed


In [3]:
# ===== Configuration =====
ROOT_DIR = Path.cwd()
EXCEL_SCHEMA = ROOT_DIR / "Standardized Seagrass Attribute Table.xlsx"
OUTPUT_DIR = ROOT_DIR / "Standardized_Output_baru"
TARGET_CRS = "EPSG:4326"

# All direct folders under ROOT_DIR will be processed, except output and system folders.
# The output preserves the source folder structure, for example:
#   Mediterranean/01_Malta/a.shp -> Standardized_Output/Mediterranean/01_Malta/a.gpkg
EXCLUDED_INPUT_DIR_NAMES = {
    OUTPUT_DIR.name.lower(),
    ".git",
    ".agents",
    ".codex",
    ".ipynb_checkpoints",
}
INPUT_DIRS = sorted(
    p for p in ROOT_DIR.iterdir()
    if p.is_dir()
    and p.name.lower() not in EXCLUDED_INPUT_DIR_NAMES
    and not p.name.lower().endswith("_standardized")
)

# Recommended default: GPKG, so long field names remain intact.
# Other option: "ESRI Shapefile". Not recommended because column names will be truncated to 10 characters.
OUTPUT_DRIVER = "GPKG"

# If the source data has no CRS, the notebook cannot reproject safely.
# Set the source CRS here if you are certain all data without a CRS uses a specific CRS, for example "EPSG:3035".
ASSUME_SOURCE_CRS_IF_MISSING = None

# If True, overwrite existing output files.
OVERWRITE = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Root:", ROOT_DIR)
print("Input folders:")
for input_dir in INPUT_DIRS:
    print(" -", input_dir.relative_to(ROOT_DIR))
print("Output:", OUTPUT_DIR)

Root: g:\Data Seagrass Global\Data Fix
Input folders:
 - Carribean
 - East Coast
 - Global
 - Indonesia
 - Mapped_Output
 - Mediterranean
 - Standardized_Output
Output: g:\Data Seagrass Global\Data Fix\Standardized_Output_baru


In [4]:
def clean_excel_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df

schema_df = pd.read_excel(EXCEL_SCHEMA, sheet_name=0)
schema_df = clean_excel_columns(schema_df)

name_col = "Column Name (snake_case)"
type_col = "Data Type"

missing_required = [c for c in [name_col, type_col] if c not in schema_df.columns]
if missing_required:
    raise ValueError(f"Required columns were not found in Excel: {missing_required}. Available columns: {list(schema_df.columns)}")

schema_df = schema_df[[name_col, type_col]].dropna(subset=[name_col]).copy()
schema_df[name_col] = schema_df[name_col].astype(str).str.strip()
schema_df[type_col] = schema_df[type_col].astype(str).str.strip()
schema_df = schema_df[schema_df[name_col].ne("")]

# Avoid duplicate column names from Excel.
schema_df = schema_df.drop_duplicates(subset=[name_col], keep="first").reset_index(drop=True)

field_specs = list(schema_df.itertuples(index=False, name=None))
print(f"Standard field count: {len(field_specs)}")
display(schema_df)

Standard field count: 35


,Column Name (snake_case),Data Type
0,data_id,String
1,country_iso,String (Alpha-3)
2,quality_tier,String (Q1-Q4)
3,status,String
4,authors_name,String
5,authors_institution,String
6,longitude_x,Float
7,latitude_y,Float
8,original_crs,String
9,depth_m,Float


In [5]:
def pandas_dtype_for_standard_type(data_type: str):
    dt = str(data_type).strip().lower()
    if dt.startswith("integer"):
        return "Int64"
    if dt.startswith("float") or dt.startswith("double") or dt.startswith("decimal"):
        return "Float64"
    if dt.startswith("date"):
        return "datetime64[ns]"
    return "string"


def add_missing_standard_fields(gdf: gpd.GeoDataFrame, specs: list[tuple[str, str]]) -> gpd.GeoDataFrame:
    gdf = gdf.copy()
    existing_lookup = {str(c).lower(): c for c in gdf.columns}
    for field_name, data_type in specs:
        # The geometry column must not be overwritten.
        if field_name.lower() in existing_lookup:
            continue
        dtype = pandas_dtype_for_standard_type(data_type)
        if dtype == "datetime64[ns]":
            gdf[field_name] = pd.NaT
            gdf[field_name] = pd.to_datetime(gdf[field_name])
        else:
            gdf[field_name] = pd.Series(pd.NA, index=gdf.index, dtype=dtype)
    return gdf

def rename_reserved_fields(gdf):
    gdf = gdf.copy()

    reserved = {
        "fid": "source_fid",
        "ogc_fid": "source_ogc_fid",
    }

    rename_map = {}

    for col in gdf.columns:
        if col == gdf.geometry.name:
            continue

        key = col.lower()
        if key in reserved:
            rename_map[col] = reserved[key]

    if rename_map:
        print("  Rename reserved fields:", rename_map)
        gdf = gdf.rename(columns=rename_map)

    return gdf

def safe_layer_name(name: str, max_len: int = 63) -> str:
    cleaned = re.sub(r"[^0-9A-Za-z_]+", "_", str(name)).strip("_")
    if not cleaned:
        cleaned = "layer"
    if cleaned[0].isdigit():
        cleaned = "layer_" + cleaned
    return cleaned[:max_len]


def shp_field_name_warning(specs: list[tuple[str, str]]):
    too_long = [name for name, _ in specs if len(name) > 10]
    if too_long:
        print("WARNING: Shapefile truncates field names longer than 10 characters. Affected fields:")
        print(too_long)


def list_gdb_layers(gdb_path: Path) -> list[str]:
    if pyogrio is not None:
        layers = pyogrio.list_layers(gdb_path)
        # pyogrio returns an array-like object. First column is layer name.
        return [str(row[0]) for row in layers]
    if fiona is not None:
        return list(fiona.listlayers(gdb_path))
    raise RuntimeError("pyogrio or fiona is required to read layers in .gdb files")


def read_vector(path: Path, layer: str | None = None) -> gpd.GeoDataFrame:
    kwargs = {}
    if layer is not None:
        kwargs["layer"] = layer
    if pyogrio is not None:
        return gpd.read_file(path, engine="pyogrio", **kwargs)
    return gpd.read_file(path, **kwargs)


def reproject_to_target(gdf: gpd.GeoDataFrame, target_crs: str, source_label: str) -> gpd.GeoDataFrame:
    if gdf.crs is None:
        if ASSUME_SOURCE_CRS_IF_MISSING is None:
            raise ValueError(
                f"{source_label} has no CRS. Set ASSUME_SOURCE_CRS_IF_MISSING if the source CRS is known."
            )
        gdf = gdf.set_crs(ASSUME_SOURCE_CRS_IF_MISSING, allow_override=True)
    return gdf.to_crs(target_crs)


def remove_existing_output(path: Path):
    if not path.exists():
        return
    if path.is_file():
        path.unlink()
    else:
        raise ValueError(f"Output already exists and is not a file: {path}")

In [6]:
# Find all shapefiles and File Geodatabases from all input folders.
shapefiles = sorted(
    shp
    for input_dir in INPUT_DIRS
    for shp in input_dir.rglob("*.shp")
)
gdbs = sorted(
    gdb
    for input_dir in INPUT_DIRS
    for gdb in input_dir.rglob("*.gdb")
    if gdb.is_dir()
)

print(f"Shapefiles found: {len(shapefiles)}")
for p in shapefiles:
    print("  SHP:", p.relative_to(ROOT_DIR))

print(f"File Geodatabases found: {len(gdbs)}")
for p in gdbs:
    print("  GDB:", p.relative_to(ROOT_DIR))
    try:
        for lyr in list_gdb_layers(p):
            print("    -", lyr)
    except Exception as exc:
        print("    Failed to read the layer list:", exc)

Shapefiles found: 143
  SHP: Carribean\01_Allan Coral Atlas FL_Keys\Benthic_Map.shp
  SHP: Carribean\01_Allan Coral Atlas FL_Keys\Geomorphic_Map.shp
  SHP: Carribean\02_Biscayne Bay Seagrass_2020_2024\ALL_DERM_SEAGRASS_2024.shp
  SHP: Carribean\02_Biscayne Bay Seagrass_2020_2024\ALL_SEAGRASS_2020_Clip.shp
  SHP: Carribean\02_Biscayne Bay Seagrass_2020_2024\ALL_SEAGRASS_2021_Clip.shp
  SHP: Carribean\02_Biscayne Bay Seagrass_2020_2024\ALL_SEAGRASS_2022_Clip.shp
  SHP: Carribean\02_Biscayne Bay Seagrass_2020_2024\ALL_SEAGRASS_2023.shp
  SHP: Carribean\03_FL_Keys_Sg_In situ\fl_keys_sg_in_situ.shp
  SHP: Carribean\04_NOAA - USFWS Habitat Maps FL_Keys\Seagrass_Habitat_in_Florida.shp
  SHP: Carribean\05_Florida_Benthic_5Feb21\Florida_Benthic_5Feb21.shp
  SHP: Carribean\07_JM_Seagrass_22Jan2025\JM_Seagrass_22Jan2025.shp
  SHP: Carribean\08_JM_Seagrass_FieldPnts_22Jan2025\JM_Seagrass_FieldPnts_22Jan2025.shp
  SHP: Carribean\09_NOAA_National Reef Monitoring program\NOAA_NCRMP_Sites_Florida.shp


In [7]:
if OUTPUT_DRIVER == "ESRI Shapefile":
    shp_field_name_warning(field_specs)

process_log = []


def output_path_for_shapefile(source_path: Path) -> Path:
    rel_parent = source_path.parent.relative_to(ROOT_DIR)
    out_dir = OUTPUT_DIR / rel_parent
    out_dir.mkdir(parents=True, exist_ok=True)
    if OUTPUT_DRIVER == "GPKG":
        return out_dir / f"{source_path.stem}.gpkg"
    if OUTPUT_DRIVER == "ESRI Shapefile":
        return out_dir / f"{source_path.stem}.shp"
    raise ValueError(f"Unsupported OUTPUT_DRIVER: {OUTPUT_DRIVER}")


def output_path_for_gdb_layer(gdb_path: Path) -> Path:
    rel_parent = gdb_path.parent.relative_to(ROOT_DIR)
    out_dir = OUTPUT_DIR / rel_parent
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir / f"{gdb_path.stem}.gpkg"


def write_output(gdf: gpd.GeoDataFrame, out_path: Path, layer_name: str | None = None):
    if OUTPUT_DRIVER == "GPKG":
        if OVERWRITE and out_path.exists() and layer_name is None:
            remove_existing_output(out_path)
        if OVERWRITE and out_path.exists() and layer_name is not None:
            # Remove the entire GPKG so old layers are not left behind from the previous run.
            remove_existing_output(out_path)
        gdf.to_file(out_path, layer=layer_name or safe_layer_name(out_path.stem), driver="GPKG")
    elif OUTPUT_DRIVER == "ESRI Shapefile":
        if OVERWRITE and out_path.exists():
            # GeoPandas/Fiona will overwrite related shapefile components if the driver supports it.
            # If this fails, manually delete the sidecar files in the output folder and rerun.
            pass
        gdf.to_file(out_path, driver="ESRI Shapefile")
    else:
        raise ValueError(f"Unsupported OUTPUT_DRIVER: {OUTPUT_DRIVER}")


# Process shapefiles one by one.
for shp in shapefiles:
    source_label = str(shp.relative_to(ROOT_DIR))
    print(f"Processing SHP: {source_label}")
    try:
        gdf = read_vector(shp)
        original_crs = str(gdf.crs) if gdf.crs is not None else None
        gdf = reproject_to_target(gdf, TARGET_CRS, source_label)
        gdf = add_missing_standard_fields(gdf, field_specs)
        gdf = rename_reserved_fields(gdf)
        out_path = output_path_for_shapefile(shp)
        if OVERWRITE and out_path.exists() and OUTPUT_DRIVER == "GPKG":
            remove_existing_output(out_path)
        write_output(gdf, out_path, layer_name=safe_layer_name(shp.stem) if OUTPUT_DRIVER == "GPKG" else None)
        process_log.append({
            "source_type": "shapefile",
            "source": source_label,
            "layer": None,
            "original_crs": original_crs,
            "target_crs": TARGET_CRS,
            "feature_count": len(gdf),
            "output": str(out_path.relative_to(ROOT_DIR)),
            "status": "ok",
            "message": "",
        })
        print("  OK ->", out_path.relative_to(ROOT_DIR), f"({len(gdf)} features)")
    except Exception as exc:
        process_log.append({
            "source_type": "shapefile",
            "source": source_label,
            "layer": None,
            "original_crs": None,
            "target_crs": TARGET_CRS,
            "feature_count": None,
            "output": None,
            "status": "failed",
            "message": str(exc),
        })
        print("  FAILED:", exc)


# Process each layer in each File Geodatabase.
for gdb in gdbs:
    source_label = str(gdb.relative_to(ROOT_DIR))
    print(f"Processing GDB: {source_label}")
    try:
        layers = list_gdb_layers(gdb)
    except Exception as exc:
        process_log.append({
            "source_type": "gdb",
            "source": source_label,
            "layer": None,
            "original_crs": None,
            "target_crs": TARGET_CRS,
            "feature_count": None,
            "output": None,
            "status": "failed",
            "message": f"Failed to list layers: {exc}",
        })
        print("  FAILED to list layers:", exc)
        continue

    out_path = output_path_for_gdb_layer(gdb)
    if OVERWRITE and out_path.exists() and OUTPUT_DRIVER == "GPKG":
        remove_existing_output(out_path)

    for layer in layers:
        print(f"  Layer: {layer}")
        try:
            gdf = read_vector(gdb, layer=layer)
            original_crs = str(gdf.crs) if gdf.crs is not None else None
            gdf = reproject_to_target(gdf, TARGET_CRS, f"{source_label} | {layer}")
            gdf = add_missing_standard_fields(gdf, field_specs)
            gdf = rename_reserved_fields(gdf)
            out_layer = safe_layer_name(layer)
            gdf.to_file(out_path, layer=out_layer, driver="GPKG")
            process_log.append({
                "source_type": "gdb",
                "source": source_label,
                "layer": layer,
                "original_crs": original_crs,
                "target_crs": TARGET_CRS,
                "feature_count": len(gdf),
                "output": f"{out_path.relative_to(ROOT_DIR)} | layer={out_layer}",
                "status": "ok",
                "message": "",
            })
            print("    OK ->", out_path.relative_to(ROOT_DIR), "layer=", out_layer, f"({len(gdf)} features)")
        except Exception as exc:
            process_log.append({
                "source_type": "gdb",
                "source": source_label,
                "layer": layer,
                "original_crs": None,
                "target_crs": TARGET_CRS,
                "feature_count": None,
                "output": None,
                "status": "failed",
                "message": str(exc),
            })
            print("    FAILED:", exc)

Processing SHP: Carribean\01_Allan Coral Atlas FL_Keys\Benthic_Map.shp
  OK -> Standardized_Output_baru\Carribean\01_Allan Coral Atlas FL_Keys\Benthic_Map.gpkg (376329 features)
Processing SHP: Carribean\01_Allan Coral Atlas FL_Keys\Geomorphic_Map.shp
  OK -> Standardized_Output_baru\Carribean\01_Allan Coral Atlas FL_Keys\Geomorphic_Map.gpkg (126571 features)
Processing SHP: Carribean\02_Biscayne Bay Seagrass_2020_2024\ALL_DERM_SEAGRASS_2024.shp
  OK -> Standardized_Output_baru\Carribean\02_Biscayne Bay Seagrass_2020_2024\ALL_DERM_SEAGRASS_2024.gpkg (486 features)
Processing SHP: Carribean\02_Biscayne Bay Seagrass_2020_2024\ALL_SEAGRASS_2020_Clip.shp
  OK -> Standardized_Output_baru\Carribean\02_Biscayne Bay Seagrass_2020_2024\ALL_SEAGRASS_2020_Clip.gpkg (541 features)
Processing SHP: Carribean\02_Biscayne Bay Seagrass_2020_2024\ALL_SEAGRASS_2021_Clip.shp
  OK -> Standardized_Output_baru\Carribean\02_Biscayne Bay Seagrass_2020_2024\ALL_SEAGRASS_2021_Clip.gpkg (474 features)
Processing 

c:\Users\asus\miniconda3\envs\geo\Lib\site-packages\pyogrio\raw.py:198: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D Polygon' is converted to 'Polygon Z'
  return ogr_read(


  OK -> Standardized_Output_baru\Global\13_seagrass_meadows_the greek seas\greek_seagrass_meadows_v0906.gpkg (9295 features)
Processing SHP: Global\14_habitat_points_all\habitat_point_all.shp
  Rename reserved fields: {'FID': 'source_fid'}
  OK -> Standardized_Output_baru\Global\14_habitat_points_all\habitat_point_all.gpkg (359021 features)
Processing SHP: Indonesia\Blue Carbon\Banggai_Kep.shp
  OK -> Standardized_Output_baru\Indonesia\Blue Carbon\Banggai_Kep.gpkg (13641 features)
Processing SHP: Indonesia\Blue Carbon\Belitung.shp
  OK -> Standardized_Output_baru\Indonesia\Blue Carbon\Belitung.gpkg (16055 features)
Processing SHP: Indonesia\Blue Carbon\Bontang.shp
  OK -> Standardized_Output_baru\Indonesia\Blue Carbon\Bontang.gpkg (6659 features)
Processing SHP: Indonesia\Blue Carbon\Kangean.shp
  OK -> Standardized_Output_baru\Indonesia\Blue Carbon\Kangean.gpkg (6028 features)
Processing SHP: Indonesia\Blue Carbon\Karimunjawa_2021_04_02.shp
  OK -> Standardized_Output_baru\Indonesia\B

c:\Users\asus\miniconda3\envs\geo\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: g:\Data Seagrass Global\Data Fix\Mediterranean\01_Malta\Seagrass_1120.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(
c:\Users\asus\miniconda3\envs\geo\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: g:\Data Seagrass Global\Data Fix\Mediterranean\01_Malta\Seagrass_1120_ETRS89.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


  OK -> Standardized_Output_baru\Mediterranean\01_Malta\Seagrass_1120.gpkg (1 features)
Processing SHP: Mediterranean\01_Malta\Seagrass_1120_ETRS89.shp
  OK -> Standardized_Output_baru\Mediterranean\01_Malta\Seagrass_1120_ETRS89.gpkg (1 features)
Processing SHP: Mediterranean\02_France Zostera\Inventaire des données des herbiers de zostères en France métropolitaine.shp


...

  OK -> Standardized_Output_baru\Mediterranean\02_France Zostera\Inventaire des données des herbiers de zostères en France métropolitaine.gpkg (324996 features)
Processing SHP: Mediterranean\03_Posidonia France\Inventaire des données d_herbier de posidonie en France métropolitaine.shp
  OK -> Standardized_Output_baru\Mediterranean\03_Posidonia France\Inventaire des données d_herbier de posidonie en France métropolitaine.gpkg (76823 features)
Processing SHP: Mediterranean\04_HABITATS_TIPUS_FONS\HABITATS_TIPUS_FONS.shp
  OK -> Standardized_Output_baru\Mediterranean\04_HABITATS_TIPUS_FONS\HABITATS_TIPUS_FONS.gpkg (49142 features)
Processing SHP: Mediterranean\05_HABITATS_MARINS_COM\HABITATS_MARINS_COM.shp
  OK -> Standardized_Output_baru\Mediterranean\05_HABITATS_MARINS_COM\HABITATS_MARINS_COM.gpkg (1392 features)
Processing SHP: Mediterranean\06_HABITATS_HABITATS_MARINS\HABITATS_HABITATS_MARINS.shp
  OK -> Standardized_Output_baru\Mediterranean\06_HABITATS_HABITATS_MARINS\HABITATS_HAB

c:\Users\asus\miniconda3\envs\geo\Lib\site-packages\pyogrio\raw.py:198: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D Polygon' is converted to 'Polygon Z'
  return ogr_read(


  OK -> Standardized_Output_baru\Mediterranean\07_Balearic isalnds\Menorca\OBSAM_BIONOMIA_MENORCA_2019.gpkg (21277 features)
Processing SHP: Mediterranean\07_Balearic isalnds\Programas_Muestreo\Puntos_Muestreos_Atlas_Posidonia.shp
  OK -> Standardized_Output_baru\Mediterranean\07_Balearic isalnds\Programas_Muestreo\Puntos_Muestreos_Atlas_Posidonia.gpkg (893 features)
Processing SHP: Mediterranean\07_Balearic isalnds\Tecnoambiente\Tecnoambiente_Bionomico_MA04_Total.shp
  OK -> Standardized_Output_baru\Mediterranean\07_Balearic isalnds\Tecnoambiente\Tecnoambiente_Bionomico_MA04_Total.gpkg (2848 features)
Processing SHP: Mediterranean\07_Balearic isalnds\Tecnoambiente\Tecnoambiente_Bionomico_MA05_Total.shp
  OK -> Standardized_Output_baru\Mediterranean\07_Balearic isalnds\Tecnoambiente\Tecnoambiente_Bionomico_MA05_Total.gpkg (3048 features)
Processing SHP: Mediterranean\07_Balearic isalnds\Tecnoambiente\Tecnoambiente_Bionomico_MA06_Total.shp
  OK -> Standardized_Output_baru\Mediterranean\

c:\Users\asus\miniconda3\envs\geo\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: g:\Data Seagrass Global\Data Fix\Mediterranean\08_Croatia 1120\1120.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


  Rename reserved fields: {'ogc_fid': 'source_ogc_fid'}
  OK -> Standardized_Output_baru\Mediterranean\08_Croatia 1120\1120.gpkg (3636 features)
Processing SHP: Mediterranean\08_Croatia 1120\cymodocea.shp
  Rename reserved fields: {'ogc_fid': 'source_ogc_fid'}
  OK -> Standardized_Output_baru\Mediterranean\08_Croatia 1120\cymodocea.gpkg (125 features)


In [8]:
log_df = pd.DataFrame(process_log)
log_path = OUTPUT_DIR / "processing_log.csv"
log_df.to_csv(log_path, index=False, encoding="utf-8-sig")

print("Ringkasan status:")
display(log_df.groupby("status", dropna=False).size().rename("count").reset_index())
print("Log saved to:", log_path)
display(log_df)

Ringkasan status:


,status,count
0,ok,143


Log saved to: g:\Data Seagrass Global\Data Fix\Standardized_Output_baru\processing_log.csv


,source_type,source,layer,original_crs,target_crs,feature_count,output,status,message
0,shapefile,Carribean\01_Allan Coral Atlas FL_Keys\Benthic...,None,EPSG:4326,EPSG:4326,376329,Standardized_Output_baru\Carribean\01_Allan Co...,ok,
1,shapefile,Carribean\01_Allan Coral Atlas FL_Keys\Geomorp...,None,EPSG:4326,EPSG:4326,126571,Standardized_Output_baru\Carribean\01_Allan Co...,ok,
2,shapefile,Carribean\02_Biscayne Bay Seagrass_2020_2024\A...,None,EPSG:4326,EPSG:4326,486,Standardized_Output_baru\Carribean\02_Biscayne...,ok,
3,shapefile,Carribean\02_Biscayne Bay Seagrass_2020_2024\A...,None,EPSG:4326,EPSG:4326,541,Standardized_Output_baru\Carribean\02_Biscayne...,ok,
4,shapefile,Carribean\02_Biscayne Bay Seagrass_2020_2024\A...,None,EPSG:4326,EPSG:4326,474,Standardized_Output_baru\Carribean\02_Biscayne...,ok,
...,...,...,...,...,...,...,...,...,...
138,shapefile,Mediterranean\07_Balearic isalnds\Tecnoambient...,None,EPSG:25831,EPSG:4326,6192,Standardized_Output_baru\Mediterranean\07_Bale...,ok,
139,shapefile,Mediterranean\07_Balearic isalnds\Tecnoambient...,None,EPSG:25831,EPSG:4326,951,Standardized_Output_baru\Mediterranean\07_Bale...,ok,
140,shapefile,Mediterranean\07_Balearic isalnds\Tecnoambient...,None,EPSG:25831,EPSG:4326,1631,Standardized_Output_baru\Mediterranean\07_Bale...,ok,
141,shapefile,Mediterranean\08_Croatia 1120\1120.shp,None,EPSG:3765,EPSG:4326,3636,Standardized_Output_baru\Mediterranean\08_Croa...,ok,


In [9]:
# Quick validation: open several outputs and check the CRS and standard fields.
# If any fields are missing, the table below will show them.

validation_rows = []
for row in process_log:
    if row["status"] != "ok" or not row["output"]:
        continue
    output_text = row["output"]
    if " | layer=" in output_text:
        file_part, layer_part = output_text.split(" | layer=", 1)
        path = ROOT_DIR / file_part
        layer = layer_part
        gdf_check = read_vector(path, layer=layer)
    else:
        path = ROOT_DIR / output_text
        gdf_check = read_vector(path)
    missing_fields = [name for name, _ in field_specs if name not in gdf_check.columns]
    validation_rows.append({
        "output": output_text,
        "crs": str(gdf_check.crs),
        "feature_count": len(gdf_check),
        "missing_standard_field_count": len(missing_fields),
        "missing_standard_fields": ", ".join(missing_fields[:20]),
    })

validation_df = pd.DataFrame(validation_rows)
display(validation_df)

if not validation_df.empty:
    bad_crs = validation_df[validation_df["crs"].ne(TARGET_CRS)]
    bad_fields = validation_df[validation_df["missing_standard_field_count"].gt(0)]
    if len(bad_crs) or len(bad_fields):
        print("Some outputs need to be checked again.")
    else:
        print("Validation OK: all outputs are readable, CRS is EPSG:4326, and all standard fields are present.")

KeyboardInterrupt: 